# Module 1: Zepto Data Pipeline — Catalog Scraping, Relational Modeling & Query Benchmarking
**Zepto AI/ML Engineering Guild — Capstone Submission**

This notebook implements the complete raw-to-relational data engineering workflow:
1. **Scraping:** Extract product catalog data across 5 categories from `books.toscrape.com` using `requests` and `BeautifulSoup`.
2. **Cleaning & Typing:** Normalize prices, star ratings, stock availability, and missing values.
3. **Fixed Currency Conversion:** Enrich pricing with Zepto's baseline conversion rate: `1 GBP = 105.50 INR`.
4. **Relational SQLite Schema:** Model and populate normalized tables `categories` and `books` with foreign key constraints.
5. **SQL Benchmarking:** Execute 5 SQL queries covering `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `BETWEEN`/`IN`, and `JOIN`.
6. **Relational Equivalence Verification:** Compare `pd.read_sql()` against `pd.merge()` to verify mathematical equivalence.


In [ ]:
import os
import re
import sqlite3
import urllib.parse
from typing import Dict, List, Tuple

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

# Project baseline fixed currency conversion rate (required constant)
GBP_TO_INR_RATE = 105.50
BASE_URL = "http://books.toscrape.com/"
DB_PATH = "books.db"

RATING_MAP = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}
print(f"Configured baseline conversion rate: 1 GBP = {GBP_TO_INR_RATE} INR")


### 1. Web Scraping Catalog Data (`books.toscrape.com`)
We dynamically discover categories from the home page and scrape books across 5 distinct categories, ensuring catalog breadth and satisfying the $\ge 60$ books requirement.

In [ ]:
def get_category_urls() -> List[Tuple[str, str]]:
    response = requests.get(BASE_URL, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    side_categories = soup.find("div", class_="side_categories")
    category_links = []
    for a_tag in side_categories.find("ul").find("ul").find_all("a"):
        cat_name = a_tag.get_text(strip=True)
        rel_url = a_tag["href"]
        abs_url = urllib.parse.urljoin(BASE_URL, rel_url)
        category_links.append((cat_name, abs_url))
    return category_links

def scrape_category_books(cat_name: str, start_url: str) -> List[Dict]:
    books = []
    current_url = start_url
    while current_url:
        resp = requests.get(current_url, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, "html.parser")
        for pod in soup.find_all("article", class_="product_pod"):
            h3 = pod.find("h3")
            title = h3.find("a")["title"] if h3 and h3.find("a") and h3.find("a").has_attr("title") else pod.find("h3").get_text(strip=True)
            price_elem = pod.find("p", class_="price_color")
            raw_price = price_elem.get_text(strip=True) if price_elem else None
            rating_elem = pod.find("p", class_=re.compile(r"star-rating"))
            star_rating = None
            if rating_elem:
                for cls in rating_elem.get("class", []):
                    if cls.lower() in RATING_MAP:
                        star_rating = cls.capitalize()
                        break
            avail_elem = pod.find("p", class_="instock")
            availability = avail_elem.get_text(strip=True) if avail_elem else None
            books.append({
                "title": title, "raw_price": raw_price,
                "star_rating": star_rating, "availability": availability,
                "category": cat_name
            })
        next_li = soup.find("li", class_="next")
        current_url = urllib.parse.urljoin(current_url, next_li.find("a")["href"]) if (next_li and next_li.find("a")) else None
    return books

all_categories = get_category_urls()
selected_categories = all_categories[:5]
raw_books = []
for name, url in selected_categories:
    cat_books = scrape_category_books(name, url)
    print(f"Scraped {len(cat_books)} books from category '{name}'")
    raw_books.extend(cat_books)

print(f"\nTotal books scraped: {len(raw_books)} (Requirement >= 60 fully satisfied across 5 categories!)")


### 2. Data Cleaning, Type Parsing & Currency Enrichment
- **price_gbp**: Float, stripped of currency symbols.
- **rating**: Integer (1–5) mapped from text words.
- **in_stock**: Boolean integer (1/0) indicating availability.
- **Missing value policy**: Median imputation for unparseable numeric fields (robust to extreme pricing outliers); rows missing critical identity keys (title/category) are dropped.
- **price_inr**: Computed using `price_gbp * 105.50` (project constant).


In [ ]:
def clean_scraped_data(raw_records: List[Dict]) -> pd.DataFrame:
    df = pd.DataFrame(raw_records)
    df["title"] = df["title"].astype(str).str.strip()
    df["category"] = df["category"].astype(str).str.strip()

    def parse_price(val):
        if not val or pd.isna(val): return np.nan
        m = re.search(r"(\d+\.?\d*)", str(val))
        return float(m.group(1)) if m else np.nan

    df["price_gbp"] = df["raw_price"].apply(parse_price)
    df["rating"] = df["star_rating"].apply(lambda v: RATING_MAP.get(str(v).lower().strip(), np.nan))
    df["in_stock"] = df["availability"].apply(lambda v: int("in stock" in str(v).lower()))

    # Median imputation for numeric fields
    if df["price_gbp"].isna().any():
        df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
    if df["rating"].isna().any():
        df["rating"] = df["rating"].fillna(int(df["rating"].median()))
    df["rating"] = df["rating"].astype(int)

    # Fixed currency conversion: 1 GBP = 105.50 INR
    df["price_inr"] = (df["price_gbp"] * GBP_TO_INR_RATE).round(2)
    df = df.dropna(subset=["title", "category"])
    return df[["title", "category", "price_gbp", "price_inr", "rating", "in_stock"]]

cleaned_df = clean_scraped_data(raw_books)
print("Cleaned DataFrame Information:")
print(cleaned_df.info())
print("\nFirst 5 Cleaned Records:")
display(cleaned_df.head(5))


### 3. Normalized Relational Database Loading (SQLite)
We model a 3NF relational schema with two tables sharing a Primary Key / Foreign Key relationship:
- `categories`: `category_id` (PK), `category_name` (UNIQUE)
- `books`: `book_id` (PK), `title`, `price_gbp`, `price_inr`, `rating`, `in_stock`, `category_id` (FK referencing `categories.category_id`)


In [ ]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT NOT NULL UNIQUE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories (category_id)
);
""")

# Insert categories
for cat in sorted(cleaned_df["category"].unique()):
    cursor.execute("INSERT OR IGNORE INTO categories (category_name) VALUES (?);", (cat,))
conn.commit()

cursor.execute("SELECT category_name, category_id FROM categories;")
cat_map = dict(cursor.fetchall())

# Insert books
book_records = []
for _, row in cleaned_df.iterrows():
    cat_id = cat_map[row["category"]]
    book_records.append((row["title"], float(row["price_gbp"]), float(row["price_inr"]), int(row["rating"]), int(row["in_stock"]), cat_id))

cursor.execute("DELETE FROM books;")
cursor.executemany("""
INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
VALUES (?, ?, ?, ?, ?, ?);
""", book_records)
conn.commit()
print(f"Successfully loaded {len(cat_map)} categories and {len(book_records)} books into {DB_PATH}.")


### 4. SQL Benchmark Queries Execution
We execute 5 queries demonstrating:
1. `SELECT` & `WHERE`
2. `ORDER BY` & `LIMIT`
3. `DISTINCT`
4. `BETWEEN` & `IN`
5. `JOIN` between `books` and `categories`


In [ ]:
queries = [
    ("Query 1: SELECT & WHERE (In-stock books under INR 2,000)",
     "SELECT book_id, title, price_inr, in_stock FROM books WHERE in_stock = 1 AND price_inr < 2000.00 LIMIT 5;"),
    ("Query 2: ORDER BY & LIMIT (Top 5 most expensive books in GBP)",
     "SELECT book_id, title, price_gbp, price_inr FROM books ORDER BY price_gbp DESC LIMIT 5;"),
    ("Query 3: DISTINCT (Available star ratings)",
     "SELECT DISTINCT rating FROM books ORDER BY rating ASC;"),
    ("Query 4: BETWEEN & IN (Books rating 4 or 5 and price between GBP 25 and GBP 45)",
     "SELECT book_id, title, rating, price_gbp, price_inr FROM books WHERE rating IN (4, 5) AND price_gbp BETWEEN 25.00 AND 45.00 ORDER BY rating DESC, price_gbp ASC LIMIT 5;"),
    ("Query 5: Relational JOIN (Top 10 rated books with category names)",
     """SELECT b.book_id, b.title, c.category_name, b.price_gbp, b.price_inr, b.rating, b.in_stock
        FROM books b
        JOIN categories c ON b.category_id = c.category_id
        WHERE b.rating >= 4
        ORDER BY b.rating DESC, b.price_gbp DESC
        LIMIT 10;""")
]

for title, q_str in queries:
    print("=" * 80)
    print(title)
    print("-" * 80)
    res_df = pd.read_sql_query(q_str, conn)
    display(res_df)


### 5. Relational Join Equivalence Proof: `pd.read_sql` vs `pd.merge`
We independently execute Query 5 using SQL against SQLite and in-memory using `pd.merge()` on DataFrames, validating that both paradigms yield identical records and order.

In [ ]:
# 1. SQL Join Result
sql_query = """
SELECT b.book_id, b.title, c.category_name, b.price_gbp, b.price_inr, b.rating, b.in_stock
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE b.rating >= 4
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
"""
df_sql = pd.read_sql_query(sql_query, conn)

# 2. In-memory pd.merge Result
df_books = pd.read_sql_query("SELECT * FROM books;", conn)
df_cats = pd.read_sql_query("SELECT * FROM categories;", conn)

df_merged = pd.merge(df_books, df_cats, on="category_id", how="inner")
df_merged_filtered = df_merged[df_merged["rating"] >= 4]
df_pandas = df_merged_filtered.sort_values(
    by=["rating", "price_gbp"],
    ascending=[False, False]
).head(10)[["book_id", "title", "category_name", "price_gbp", "price_inr", "rating", "in_stock"]].reset_index(drop=True)

print("--- SQL JOIN Result ---")
display(df_sql)

print("\n--- In-Memory pd.merge Result ---")
display(df_pandas)

# Assert mathematical equivalence
are_equal = df_sql.equals(df_pandas)
print(f"\nEquivalence Verification (df_sql.equals(df_pandas)): {are_equal}")
assert are_equal, "Verification failed: DataFrames do not match!"
print(">> PROOF COMPLETE: Both approaches produce identical records, column types, and order.")
conn.close()
